<a href="https://colab.research.google.com/github/Makayla-Kelly/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U duckdb huggingface_hub

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of visible SQL/code
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"
CLI = f"read_parquet('{REL}/dim_clients.parquet')"

print(" Connected to FlyRank warehouse")
print("Development month: March 2026")

 Connected to FlyRank warehouse
Development month: March 2026


In [ ]:
march_schema = con.sql(f"""
    SELECT *
    FROM {MAR}
    LIMIT 0
""")

print(march_schema.columns)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

What one row means:
One row represents one content item for one client on one reporting date.

Which table(s) I will use:
I will mainly use fact_content_daily_performance. I may use dim_content for stable content metadata, but the daily performance table is the main source for this lane.

Time window:
I will use March 2026 as my development month. June 2026 will not be used for developing the label or feature logic because it is the final outcome month.

What I will predict or rank:
I will rank content items by refresh opportunity, meaning which pages appear most worth reviewing or updating based on their search and traffic performance.

What I deliberately exclude:
I will exclude future information, label-derived fields, IDs as predictive features, and any existing FlyRank decision flag that would directly reveal the action I am trying to predict.

In [ ]:
dim_schema = con.sql(f"""
    SELECT *
    FROM {DIM}
    LIMIT 0
""")

print(dim_schema.columns)

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


## 2. Fields
Features:
impressions_31d: observed March search visibility available at the decision moment.
clicks_31d: observed March search traffic available at the decision moment.
ctr_31d: calculated from March clicks and impressions, so it is available at the decision moment.
avg_position_31d: observed March average search position available at the decision moment.
gsc_days_observed: number of March days with available GSC data, known at the decision moment.
Label / proxy:
refresh_opportunity: a future performance-change indicator based on April 2026 clicks. It represents whether a content item experienced a substantial decline after the March feature window. It is used only as the outcome and never as an input feature.
Context:
client_hash_id and content_hash_id are used to identify and group content items. report_date is used to define the observation window. These fields are not model features.
Excluded:
Future-period performance measures, label-derived fields, IDs as predictors, and existing FlyRank decision flags are excluded because they could reveal the outcome or leak existing decision logic into the model.

In [ ]:
grain_check = con.sql(f"""
WITH grain_counts AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {MAR}
    GROUP BY 1, 2, 3
)
SELECT
    SUM(n) AS total_rows,
    COUNT(*) AS unique_grain_rows,
    SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END) AS duplicate_grain_groups
FROM grain_counts
""").df()

grain_check

,total_rows,unique_grain_rows,duplicate_grain_groups
0,9841378.0,9841378,0.0


Result: The March slice contains 9,841,378 rows and 9,841,378 unique date-client-content combinations, with 0 duplicate grain groups. This confirms that one row represents one content item for one client on one reporting date.

## 3. Verify it with queries (grain, counts, missing values, windows)

In [ ]:
slice_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
""").df()

slice_check

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
availability_check = con.sql(f"""
SELECT
    (SELECT COUNT(*) FROM {MAR}) AS total_rows,
    (
        SELECT COUNT(*)
        FROM {MAR}
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    ROUND(
        100.0 *
        (
            SELECT COUNT(*)
            FROM {MAR}
            WHERE gsc_data_available IS TRUE
        )
        /
        (SELECT COUNT(*) FROM {MAR}),
        2
    ) AS percent_surviving
""").df()

availability_check

,total_rows,gsc_available_rows,percent_surviving
0,9841378,3611061,36.69


Result: Of 9,841,378 March rows, 3,611,061 have GSC data available when filtering with gsc_data_available IS TRUE. This leaves 36.69% of the March slice for analyses that require Search Console signals.

Five-feature frame: March 2026

In [ ]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions_31d,
    SUM(gsc_clicks) AS clicks_31d,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr_31d,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS avg_position_31d,

    COUNT(*) AS gsc_days_observed

FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
HAVING SUM(gsc_impressions) > 0
""").df()

print("Feature-frame shape:", feature_frame.shape)
feature_frame.head()

Feature-frame shape: (176738, 7)


,client_hash_id,content_hash_id,impressions_31d,clicks_31d,ctr_31d,avg_position_31d,gsc_days_observed
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,0.001754,4.450877,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,0.000000,2.298246,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,0.000000,5.637584,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,0.004222,6.906404,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,0.005776,3.950542,31


impressions_31d, knowable at the decision moment because it uses only observed March impressions.
clicks_31d, knowable at the decision moment because it uses only observed March clicks.
ctr_31d, knowable at the decision moment because it is calculated only from March clicks and impressions.
avg_position_31d, knowable at the decision moment because it uses only observed March search-position data.
gsc_days_observed, knowable at the decision moment because it counts only March days for which GSC data was available.

In [ ]:
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

In [ ]:
april_outcomes = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS april_clicks_31d
FROM {APR}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

april_outcomes.head()

,client_hash_id,content_hash_id,april_clicks_31d
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,0.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,0.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,0.0


In [ ]:
model_frame = feature_frame.merge(
    april_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_frame["refresh_opportunity"] = (
    model_frame["april_clicks_31d"]
    < model_frame["clicks_31d"] * 0.80
).astype(int)

print("Rows with March features + April outcome:", len(model_frame))
print("\nLabel counts:")
print(model_frame["refresh_opportunity"].value_counts())

print("\nLabel rate:")
print(model_frame["refresh_opportunity"].value_counts(normalize=True).round(3))

Rows with March features + April outcome: 158549

Label counts:
refresh_opportunity
0    117496
1     41053
Name: count, dtype: int64

Label rate:
refresh_opportunity
0    0.741
1    0.259
Name: proportion, dtype: float64


Label timing: refresh_opportunity is calculated using April 2026 performance, after the March feature window. April performance is therefore used only to construct the outcome and is not an allowed predictor.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

FEATURES = [
    "impressions_31d",
    "clicks_31d",
    "ctr_31d",
    "avg_position_31d",
    "gsc_days_observed"
]

X = model_frame[FEATURES]
y = model_frame["refresh_opportunity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

honest_model.fit(X_train, y_train)

honest_probs = honest_model.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, honest_probs)

print(f"Honest ROC-AUC: {honest_auc:.3f}")

Honest ROC-AUC: 0.862


In [ ]:
# Deliberate leakage: copy the label into a predictor
model_frame["leaky_label_copy"] = model_frame["refresh_opportunity"]

LEAKY_FEATURES = FEATURES + ["leaky_label_copy"]

X_leaky = model_frame[LEAKY_FEATURES]
y_leaky = model_frame["refresh_opportunity"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky,
    y_leaky,
    test_size=0.20,
    random_state=42,
    stratify=y_leaky
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

leaky_model.fit(X_train_l, y_train_l)

leaky_probs = leaky_model.predict_proba(X_test_l)[:, 1]
leaky_auc = roc_auc_score(y_test_l, leaky_probs)

print(f"Honest ROC-AUC: {honest_auc:.3f}")
print(f"Leaky ROC-AUC:  {leaky_auc:.3f}")

Honest ROC-AUC: 0.862
Leaky ROC-AUC:  1.000


In [ ]:
model_frame.drop(columns=["leaky_label_copy"], inplace=True)

print("Leakage column removed.")
print("leaky_label_copy" in model_frame.columns)
print(f"Final honest ROC-AUC retained: {honest_auc:.3f}")

Leakage column removed.
False
Final honest ROC-AUC retained: 0.862


Leakage lesson: The honest model uses only information available during the March decision window. I then deliberately added leaky_label_copy, which was directly derived from the April outcome label. The model score jumped toward perfect because the predictor contained the answer itself. I removed the leaked column and retained the honest ROC-AUC as the valid score.

## 4. Data limits

Limitation: GSC coverage is incomplete in this slice. Only 3,611,061 of 9,841,378 March rows, or 36.69%, have gsc_data_available IS TRUE. This means the feature frame represents only content with usable Search Console observations and may not represent clients or content with missing GSC history. The refresh-opportunity label is also a simple proxy based on future click decline rather than a confirmed business outcome.

Result: The honest five-feature model achieved a ROC-AUC of 0.862. After intentionally adding a direct copy of the outcome label as a predictor, ROC-AUC increased to 1.000. This apparent improvement is entirely caused by target leakage and would not be available at a real decision moment. I removed the leaked feature and retained 0.862 as the valid score.